#Initialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import trim, col, regexp_replace, substring, length, to_date, abs
from pyspark.sql.window import Window

# Reading From Bronze

In [0]:
df = spark.table("salesdatalakehouse.bronze.crm_sales_details_raw")

# Data Transformations

## Replace dashes with underscores

In [0]:
# erp table for categories contain '_'

df = df.withColumn("sls_prd_key", regexp_replace(col("sls_prd_key"), "-", "_"))

##Handle Dates

In [0]:
# order date

df = (df
.withColumn(
    "sls_order_dt",
    F.when((length(col("sls_order_dt").cast("string")) != 8), None)
    .when(col("sls_order_dt") <= 0, None)
    .otherwise(col("sls_order_dt"))
)
)

####Convert to date datatype

In [0]:

int_date_cols = [
    c for c, t in df.dtypes 
    if t in ("int", "bigint", "long") 
    and any(keyword in c.lower() for keyword in ["dt", "date"])
]
for i in int_date_cols:
    df = df.withColumn(i, to_date(col(i).cast("string"), "yyyyMMdd"))

## Handle Sales, Quantity and Price

In [0]:

df = (
    df
# when sales is negative, zero or null, derive using quantity and price
      .withColumn("sls_sales", 
                  F.when((col("sls_sales") <= 0 )| 
                         (col("sls_sales").isNull()) | 
                         (col("sls_sales") != (col("sls_quantity") * abs(col("sls_price")))), 
                         (col("sls_quantity") * abs(col("sls_price"))))
                  .otherwise(col("sls_sales"))
                  )
# when price is negative, zero or null, derive using quantity and sales
     .withColumn("sls_price", 
                  F.when((col("sls_price") <= 0 )| 
                         (col("sls_price").isNull()) | 
                         (abs(col("sls_price")) != (abs(col("sls_sales")) / abs(col("sls_quantity")))), 
                         (abs(col("sls_sales")) / 
                          F.when(abs(col("sls_quantity")) == 0, None).otherwise(abs(col("sls_quantity")))
                          ))
                  .otherwise(col("sls_price"))
                  )
     )


## Renaming Colmns

In [0]:
rename_map = {
    "sls_ord_num"  : "order_number",
    "sls_prd_key"  : "product_key",
    "sls_cust_id"  : "customer_id",
    "sls_order_dt" : "order_date",
    "sls_ship_dt"  : "ship_date",
    "sls_due_dt"   : "due_date",
    "sls_sales"    : "sales_amount",
    "sls_quantity" : "quantity",
    "sls_price"    : "price"
}

for old_name, new_name in rename_map.items():
    df = df.withColumnRenamed(old_name, new_name)

# Load Into Silver

In [0]:
df.write.format("delta").mode("overwrite").option('overwriteSchema', 'true').saveAsTable('salesdatalakehouse.silver.crm_sales_details')